# One-step-ahead LSTM

A compact rework of the original LSTM experiment. The scaler is fitted on training data only, and each prediction uses the previous 24 *observed* hourly values. This design evaluates one-step-ahead predictions. The reconstructed CSV is included, but model metrics have not been re-executed or verified.


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

tf.keras.utils.set_random_seed(42)


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [Path("../data/clean_hourly_data.csv"), Path("data/clean_hourly_data.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("The reconstructed CSV is missing; see data/README.md")

data = pd.read_csv(data_path, parse_dates=["ds"])
required = {"ds", "y"}
if not required.issubset(data.columns):
    raise ValueError("The CSV must contain ds (timestamp) and y (numeric target) columns")
data = data[["ds", "y"]].copy().sort_values("ds").reset_index(drop=True)
data["y"] = pd.to_numeric(data["y"], errors="raise")
if data.isna().any().any() or data["ds"].duplicated().any():
    raise ValueError("Remove missing values and duplicate timestamps before running these notebooks")
if not data["ds"].diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("These notebooks require consecutive hourly observations")

train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)
if train_end < 48 or validation_end - train_end < 24 or len(data) - validation_end < 24:
    raise ValueError("The dataset is too short for the chronological split")
print(f"Rows: {len(data):,}; train: {train_end:,}; validation: {validation_end - train_end:,}; test: {len(data) - validation_end:,}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def report(y_true, y_pred, timestamps, title):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{title} | MAE: {mae:.3f} | RMSE: {rmse:.3f}")
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(timestamps, y_true, label="Observed", linewidth=1)
    ax.plot(timestamps, y_pred, label="Predicted", linewidth=1)
    ax.set(title=title, xlabel="Time", ylabel="Hourly sum of minute-level kW")
    ax.legend()
    fig.tight_layout()
    plt.show()


## Scale and build chronological windows


In [ ]:
lookback = 24
scaler = MinMaxScaler()
scaler.fit(data[["y"]].iloc[:train_end])
scaled = scaler.transform(data[["y"]]).astype("float32").ravel()

X = np.stack([scaled[i - lookback:i] for i in range(lookback, len(scaled))])
y = scaled[lookback:]
target_rows = np.arange(lookback, len(data))
train_mask = target_rows < train_end
validation_mask = (target_rows >= train_end) & (target_rows < validation_end)
test_mask = target_rows >= validation_end
X = X[..., np.newaxis]
print("Window tensor:", X.shape)


## Train on the first 70%, stop on validation


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(lookback, 1)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse")
history = model.fit(
    X[train_mask], y[train_mask],
    validation_data=(X[validation_mask], y[validation_mask]),
    epochs=30, batch_size=64, shuffle=False, verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    )],
)


## Evaluate the untouched test period


In [ ]:
predicted_scaled = model.predict(X[test_mask], verbose=0)
predicted = scaler.inverse_transform(predicted_scaled).ravel()
observed = data.loc[target_rows[test_mask], "y"].to_numpy()
previous_hour = data.loc[target_rows[test_mask] - 1, "y"].to_numpy()
print(f"Previous-hour baseline MAE: {mean_absolute_error(observed, previous_hour):.3f}")
report(observed, predicted, data.loc[target_rows[test_mask], "ds"], "LSTM: observed 24-hour window")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(history.history["loss"], label="Train")
ax.plot(history.history["val_loss"], label="Validation")
ax.set(xlabel="Epoch", ylabel="Scaled MSE", title="Training history")
ax.legend()
fig.tight_layout()
plt.show()
